## XML to relational table

Extract colonist data into two normalized DataFrames:
- **pawns** — one row per colonist (`pawn_id`, `first`, `last`)
- **skills** — one row per skill entry (`pawn_id`, `skill`, `level`)

In [2]:
import xml.etree.ElementTree as ET
import pandas as pd

tree = ET.parse("logs/09_04_2026.rws")
root = tree.getroot()

### 1. Build the `pawns` table
- One row per colonist.
- Columns: `pawn_id` (auto-assigned integer), `first`, `last`.

- Use the same XPath from exercise 11 to get the colonist list
- `pawn_id` must be assigned manually — there's a built-in that gives you both the index and value while iterating
- use the method that safely returns `None` when the tag is missing (instead of `.find().text`)
- Build a dict per row, collect them in a list, then pass to `pd.DataFrame()`

In [2]:
things = root.find("game/maps/li/things")

pawn_list = []

pawns = things.findall(".//thing[@Class='Pawn'][kindDef='Colonist']")
for pawn_id, pawn in enumerate(pawns):
    pawn_list.append({
        'first': pawn.findtext('name/first'),
        'last': pawn.findtext('name/last')
    })

df = pd.DataFrame(pawn_list)

print(df)

   first   last
0     티코   크레이그
1    나나미  커즈이스크
2    앤소니   크레이그
3     콜린  하이드로넷
4    다프네  하이드로넷
5   루드밀라     피셔
6    히스인   라우이트
7    헤스크  커즈이스크
8   베네딕트     노턴
9    마나부  하이드로넷
10  히스이스   윈'하욘
11   마룰로  하이드로넷
12   피노보    호네본
13   이리나   크레이그
14  그레이시   플로레스
15  그레고리  하이드로넷


### 2. Build the `skills` table
- One row per skill per colonist.
- Columns: `pawn_id` (FK → pawns), `skill`, `level`.
- Skills omitted from XML mean level 0 — fill them in explicitly.

**Hints:**
- Level-0 skills are absent from the XML — define a constant list of all 12 skills and use it as your reference
- Build a `{skill_name: level}` dict from each pawn's XML entries — it makes filling in missing skills straightforward
- Assign `pawn_id` the same way as in exercise 1 so the two tables can be `merge`d later

In [10]:
ALL_SKILLS = [
    "Shooting", "Melee", "Construction", "Mining", "Cooking", 
    "Plants", "Animals", "Crafting", "Artistic", "Medicine",
    "Social", "Intellectual"
]

things = root.find("game/maps/li/things")
pawns = things.findall(".//thing[@Class='Pawn'][kindDef='Colonist']")

skill_list = []

for pawn_id, pawn in enumerate(pawns):
    skills = {
        skill.findtext('def'): skill.findtext('level') or 0
        for skill in pawn.findall("skills/skills/li")
    }

    for skill_name in ALL_SKILLS:
        skill_list.append({
            'pawn_id': pawn_id,
            'skill': skill_name,
            'level': skills.get(skill_name, 0)
        })

df = pd.DataFrame(skill_list)
print(df)

     pawn_id         skill level
0          0      Shooting     8
1          0         Melee     1
2          0  Construction     1
3          0        Mining     6
4          0       Cooking     8
..       ...           ...   ...
187       15      Crafting     0
188       15      Artistic     0
189       15      Medicine     0
190       15        Social     0
191       15  Intellectual     0

[192 rows x 3 columns]
